In [ ]:
"""
Image recognition example (binary: digit '0' vs digit '1')
using synthetic 20x20 grayscale image-like data.

Demonstrates THREE ways to compute a Dense layer's forward pass,
all verified to produce the SAME output because they use the
SAME trained weights (extracted from the Keras model):

  1. Keras Sequential model (model.predict)
  2. Manual NumPy loop-based dense layer (my_dense)
  3. Manual NumPy vectorized dense layer (my_dense_v)
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

np.random.seed(42)
tf.random.set_seed(42)

# ---------------------------------------------------------------
# 1. Generate synthetic image-like data
# ---------------------------------------------------------------
# We fake "0" and "1" digit images as 20x20 (=400 pixel) grayscale
# arrays. Real digit datasets (like the course's) have actual pixel
# structure; here we just create two distinguishable random pixel
# distributions so the network has something learnable.
#
#   - Class 0 images: pixel values centered around a LOW mean
#   - Class 1 images: pixel values centered around a HIGH mean
# This mimics "0 digits look darker/lighter than 1 digits" just
# enough for a small network to learn a real decision boundary.

m = 1000  # number of examples
n = 400   # 20x20 flattened pixels

X0 = np.random.normal(loc=0.1, scale=0.05, size=(m // 2, n))  # near-black
X1 = np.random.normal(loc=0.9, scale=0.05, size=(m // 2, n))  # near-white
X = np.clip(np.vstack([X0, X1]), 0, 1)    
                 # (m, 400)
y0 = np.zeros((m // 2, 1))
y1 = np.ones((m // 2, 1))
y = np.vstack([y0, y1])                                       # (m, 1)

# Shuffle so classes are mixed, not in two solid blocks
shuffle_idx = np.random.permutation(m)
X, y = X[shuffle_idx], y[shuffle_idx]

print(f"X shape: {X.shape}, y shape: {y.shape}")


# ---------------------------------------------------------------
# 2. Visualize a random sample of the synthetic "images"
# ---------------------------------------------------------------
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1)
for ax in axes.flat:
    random_index = np.random.randint(m)
    X_random_reshaped = X[random_index].reshape((20, 20)).T
    ax.imshow(X_random_reshaped, cmap='gray')
    ax.set_title(int(y[random_index, 0]))
    ax.set_axis_off()
fig.suptitle("Sample synthetic images (label)", fontsize=14)
plt.show()


# ---------------------------------------------------------------
# 3. Build and train the Keras Sequential model
# ---------------------------------------------------------------
# Same 400 -> 25 -> 15 -> 1 architecture as your exercise, all
# sigmoid activations, binary crossentropy loss.

model = Sequential(
    [
        tf.keras.Input(shape=(400,)),
        Dense(25, activation='sigmoid'),
        Dense(15, activation='sigmoid'),
        Dense(1, activation='sigmoid'),
    ],
    name="my_model"
)

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.01),
)

model.fit(X, y, epochs=50, verbose=0)
print("Training complete.")

model.summary()


# ---------------------------------------------------------------
# 4. Keras predictions (Method 1)
# ---------------------------------------------------------------
prediction0 = model.predict(X[0].reshape(1, 400), verbose=0)
print(f"Keras predicting sample 0 (label={int(y[0,0])}): {prediction0}")


# ---------------------------------------------------------------
# 5. Extract the trained weights so all 3 methods use IDENTICAL
#    parameters -- this is what makes the comparison meaningful.
# ---------------------------------------------------------------
W1, b1 = model.layers[0].get_weights()   # 400x25, 25
W2, b2 = model.layers[1].get_weights()   # 25x15,  15
W3, b3 = model.layers[2].get_weights()   # 15x1,   1

print(f"W1 shape: {W1.shape}, b1 shape: {b1.shape}")
print(f"W2 shape: {W2.shape}, b2 shape: {b2.shape}")
print(f"W3 shape: {W3.shape}, b3 shape: {b3.shape}")


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# ---------------------------------------------------------------
# 6. Method 2: Manual NumPy loop-based dense layer (my_dense)
# ---------------------------------------------------------------
# Computes ONE example at a time, one unit at a time -- this is
# the "textbook" implementation that mirrors the math directly.

def my_dense(a_in, W, b, g):
    """
    Args:
      a_in (n,)   : single example, n features
      W    (n,j)  : weight matrix, n inputs -> j units
      b    (j,)   : bias vector, j units
      g            : activation function
    Returns:
      a_out (j,)  : output of this layer
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        w = W[:, j]                       # weights for unit j (column j)
        z = np.dot(w, a_in) + b[j]
        a_out[j] = g(z)
    return a_out


def my_sequential(x, W1, b1, W2, b2, W3, b3):
    """Chains 3 my_dense calls -- manual forward pass for ONE example."""
    a1 = my_dense(x, W1, b1, sigmoid)
    a2 = my_dense(a1, W2, b2, sigmoid)
    a3 = my_dense(a2, W3, b3, sigmoid)
    return a3


# ---------------------------------------------------------------
# 7. Method 3: Manual NumPy vectorized dense layer (my_dense_v)
# ---------------------------------------------------------------
# Computes ALL examples in a matrix at once -- this is how Keras
# actually does it internally (no Python for-loop over units).

def my_dense_v(A_in, W, b, g):
    """
    Args:
      A_in (m,n)  : m examples, n features each
      W    (n,j)  : weight matrix
      b    (j,)   : bias vector (broadcasts across m rows)
      g            : activation function
    Returns:
      A_out (m,j) : m examples, j units
    """
    z = np.matmul(A_in, W) + b
    A_out = g(z)
    return A_out


def my_sequential_v(X, W1, b1, W2, b2, W3, b3):
    """Chains 3 my_dense_v calls -- vectorized forward pass for ALL examples."""
    A1 = my_dense_v(X, W1, b1, sigmoid)
    A2 = my_dense_v(A1, W2, b2, sigmoid)
    A3 = my_dense_v(A2, W3, b3, sigmoid)
    return A3


# ---------------------------------------------------------------
# 8. Compare all three methods on the SAME example
# ---------------------------------------------------------------
sample_idx = 0
x_sample = X[sample_idx]

keras_pred = model.predict(x_sample.reshape(1, 400), verbose=0)[0, 0]
loop_pred = my_sequential(x_sample, W1, b1, W2, b2, W3, b3)[0]
vector_pred = my_sequential_v(x_sample.reshape(1, 400), W1, b1, W2, b2, W3, b3)[0, 0]

print("\n--- Comparing all 3 methods on the same example ---")
print(f"True label:              {int(y[sample_idx, 0])}")
print(f"Keras model.predict:     {keras_pred:.6f}")
print(f"my_dense (loop):         {loop_pred:.6f}")
print(f"my_dense_v (vectorized): {vector_pred:.6f}")
print("All three should match (small floating point differences are normal).")


# ---------------------------------------------------------------
# 9. Full-dataset predictions with the vectorized method + accuracy
# ---------------------------------------------------------------
Yhat = my_sequential_v(X, W1, b1, W2, b2, W3, b3)
Yhat_labels = (Yhat >= 0.5).astype(int)
accuracy = np.mean(Yhat_labels == y)
print(f"\nManual NumPy (vectorized) accuracy on training data: {accuracy:.4f}")


# ---------------------------------------------------------------
# 10. Visualize predictions: label vs Keras yhat vs manual NumPy yhat
# ---------------------------------------------------------------
fig, axes = plt.subplots(8, 8, figsize=(8, 8))
fig.tight_layout(pad=0.1, rect=[0, 0.03, 1, 0.92])
for ax in axes.flat:
    random_index = np.random.randint(m)
    X_random_reshaped = X[random_index].reshape((20, 20)).T
    ax.imshow(X_random_reshaped, cmap='gray')

    my_prediction = my_sequential(X[random_index], W1, b1, W2, b2, W3, b3)
    my_yhat = int(my_prediction[0] >= 0.5)

    tf_prediction = model.predict(X[random_index].reshape(1, 400), verbose=0)
    tf_yhat = int(tf_prediction[0, 0] >= 0.5)

    ax.set_title(f"{int(y[random_index,0])},{tf_yhat},{my_yhat}")
    ax.set_axis_off()
fig.suptitle("Label, yhat Tensorflow, yhat Numpy", fontsize=16)
plt.show()


# ---------------------------------------------------------------
# 11. Show a misclassified example (if any exist)
# # ---------------------------------------------------------------
# errors = np.where(y != Yhat_labels)[0]
# if len(errors) > 0:
#     random_index = errors[0]
#     fig = plt.figure(figsize=(2, 2))
#     X_random_reshaped = X[random_index].reshape((20, 20)).T
#     plt.imshow(X_random_reshaped, cmap='gray')
#     plt.title(f"Label={int(y[random_index,0])}, Yhat={int(Yhat_labels[random_index,0])}")
#     plt.axis('off')
#     plt.show()
# else:
#     print("No misclassifications found on this synthetic dataset.")